# SQL or Custom Expressions?

SLayer draws a clean line between **SQL** and its **DSL**:

- **Model definitions** (dimensions, measures, filters) use **raw SQL** — they reference the underlying table columns directly.
- **Queries** use the **DSL** — they reference dimension/measure names, apply transforms like `cumsum` and `change`, and filter on named entities.

The model is the abstraction boundary between the two. This notebook demonstrates the distinction with working examples.

See also: [SQL vs DSL](sql_vs_dsl.md) | [Formulas](../../concepts/formulas.md) | [Models](../../concepts/models.md) | [Queries](../../concepts/queries.md)

**Prerequisites:** `pip install motley-slayer[examples]` (jafgen is installed by the cell below)

In [1]:
# Install jafgen (Jaffle Shop data generator) from a specific commit
# The released PyPI version has a bug; this pinned commit is the fix.
# This is only needed for running the tutorials — not a SLayer dependency.
!pip install -q git+https://github.com/rossbowen/jaffle-shop-generator.git@09557a1118b000071f8171aa97d54d5029bf0f0b


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

## Model Definitions Use SQL

A model's dimensions and measures define the mapping from semantic names to SQL expressions. The `sql` field is a raw SQL column reference or expression — it's what ends up in the generated `SELECT` and `GROUP BY`.

For example, the `orders` model maps the dimension name `order_total` to the SQL column `order_total`, and the measure name `order_total` to an aggregation over `order_total`:

In [3]:
orders_model = next(m for m in models if m.name == "orders")

print("Dimensions (name -> SQL expression):")
for dim in orders_model.dimensions[:7]:  # own columns only
    print(f"  {dim.name:<20} -> sql: {dim.sql!r:<20} type: {dim.type}")

print("\nMeasures (name -> SQL expression):")
for m in orders_model.measures[:6]:
    sql_str = m.sql or "(*)"
    print(f"  {m.name:<20} -> sql: {sql_str!r:<20}")

Dimensions (name -> SQL expression):
  id                   -> sql: 'id'                 type: string
  customer_id          -> sql: 'customer_id'        type: string
  ordered_at           -> sql: 'ordered_at'         type: date
  store_id             -> sql: 'store_id'           type: string

Measures (name -> SQL expression):
  subtotal             -> sql: 'subtotal'          
  tax_paid             -> sql: 'tax_paid'          
  order_total          -> sql: 'order_total'       
  ordered_at           -> sql: 'ordered_at'        


Notice that the dimension/measure `name` is a semantic label, while `sql` is the raw SQL expression. They often coincide (e.g., `name: "order_total"`, `sql: "order_total"`), but they don't have to — a dimension could be named `revenue` with `sql: "amount * quantity"`.

Model-level filters are also raw SQL:

```yaml
filters:
  - "deleted_at IS NULL"
  - "status <> 'test'"
```

## Queries Use the DSL

When you write a query, you reference **dimension and measure names** — the semantic side. You never write SQL in a query.

Transform functions like `cumsum`, `change`, and `time_shift` are part of the DSL. They compile to complex SQL (CTEs, window functions, self-joins), but you just name them:

In [4]:
result = engine.execute(
    query={
        "source_model": "orders",
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "fields": [
            "order_total:sum",
            {"formula": "change(order_total:sum)", "name": "mom_change"},
        ],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 6,
        "offset": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>12} {'MoM Change':>12}")
print("-" * 38)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    print(f"{month:<12} ${rev:>11,.2f} {chg_str:>12}")

print("\n SQL of the query", result.sql)

Month             Revenue   MoM Change
--------------------------------------
2019-09      $  36,416.40     $+12,406
2019-10      $  70,540.19     $+34,124
2019-11      $  69,639.88        $-900
2019-12      $  74,727.42      $+5,088
2020-01      $  77,483.46      $+2,756
2020-02      $  71,937.08      $-5,546

 SQL of the query WITH base AS (
SELECT
  DATE_TRUNC('MONTH', orders.ordered_at) AS "orders.ordered_at",
  SUM(orders.order_total) AS "orders.order_total_sum"
FROM orders AS orders
GROUP BY
  DATE_TRUNC('MONTH', orders.ordered_at)
),
base_rn AS (
SELECT "orders.order_total_sum", "orders.ordered_at", ROW_NUMBER() OVER (ORDER BY "orders.ordered_at") AS _rn FROM base
),
shifted_base_mom_change AS (
SELECT
  DATE_TRUNC('MONTH', orders.ordered_at) AS "orders.ordered_at",
  SUM(orders.order_total) AS "orders.order_total_sum"
FROM orders AS orders
GROUP BY
  DATE_TRUNC('MONTH', orders.ordered_at)
),
shifted_mom_change AS (
SELECT "orders.order_total_sum", "orders.ordered_at", ROW_NUMBE

The `change()` function compiled to a self-join CTE under the hood — but the query only referenced the measure name `order_total:sum` and the transform name `change`. No SQL needed to be written by the caller.

## Filters in Queries Reference Model Entities

Query-level filters use dimension and measure names — including dotted names for joined dimensions. SLayer resolves them to the correct SQL automatically:

In [5]:
# Filter on a joined dimension name — not a SQL column reference
result = engine.execute(
    query={
        "source_model": "orders",
        "fields": ["*:count", "order_total:sum"],
        "dimensions": ["stores.name"],
        "filters": ["stores.name in ('Brooklyn', 'Philadelphia')"],
    }
)

for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders._count']:,} orders, ${row['orders.order_total_sum']:,.2f}")

print("\nThe filter 'stores.name' resolved to SQL:")
# Show just the WHERE clause from the generated SQL
where_line = [line for line in result.sql.split("\n") if "WHERE" in line or " IN" in line]
for line in where_line:
    print(f"  {line.strip()}")

  Philadelphia: 59,138 orders, $766,555.51
  Brooklyn: 83,419 orders, $1,068,207.08

The filter 'stores.name' resolved to SQL:
  WHERE
  stores.name IN ('Brooklyn', 'Philadelphia')


## Adding SQL-Level Filters via ModelExtension

What if you need a filter that references raw SQL — an expression over underlying columns that can't be reduced to a combination of dimension/measure filters?

Use query-time model extension semantics to add the filter to the **model definition** inline, right inside the query. Model-level filters are SQL, so any valid SQL expression works:

In [6]:
# "subtotal > tax_paid * 5" is raw SQL — it references underlying table columns,
# not dimension names. So it goes on the model (via ModelExtension), not the query.
result = engine.execute(
    query={
        "source_model": {
            "source_name": "orders",
            "filters": ["subtotal > tax_paid * 5"],
        },
        "fields": ["*:count", "order_total:sum"],
        "dimensions": ["stores.name"],
        "order": [{"column": "order_total_sum", "direction": "desc"}],
    }
)

print("Orders where subtotal > 5x tax (raw SQL filter via ModelExtension):")
for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders._count']:,} orders, ${row['orders.order_total_sum']:,.2f}")

Orders where subtotal > 5x tax (raw SQL filter via ModelExtension):
  Brooklyn: 83,419 orders, $1,068,207.08
  Philadelphia: 59,138 orders, $766,555.51
  Chicago: 31,722 orders, $422,840.31
  San Francisco: 27,742 orders, $370,138.50
  New Orleans: 4,115 orders, $54,693.56


The key insight: the raw SQL filter `subtotal > tax_paid * 5` is on the **model** (via `ModelExtension`), while the query's own `fields`, `dimensions`, and any query-level `filters` stay in DSL territory.

## Query Result as Model

What if you want to use DSL transforms like `time_shift` or `change` to define derived measures or dimensions?

Use `create_model()` with a `query` parameter to save a query's result as a permanent model. The derived columns become dimensions and measures on the new model, which can then be queried like any other - or just use a query inside another query, as covered in [multistage queries](../06_multistage_queries/multistage_queries.md)

See [Creating Models from Queries](../../concepts/models.md#creating-models-from-queries) for details.

## Summary

| Layer | Language | Examples |
|-------|----------|----------|
| **Model definitions** (dimensions, measures, filters) | Raw SQL | `sql: "amount"`, `filters: ["deleted_at IS NULL"]` |
| **Queries** (fields, dimensions, filters) | DSL names | `{"formula": "change(revenue)"}`, `filters: ["status = 'active'"]` |
| **SQL in queries** | ModelExtension | Add raw SQL filters/dimensions to the model inline |
| **DSL in models** | Query-as-model | Save a DSL query result as a permanent model |